# Bengali NLP Safety — V4, revised using your actual V3 results

**Your best measured Kaggle score is still V2: 0.516. V3 scored 0.446. This revised V4 has not been scored on Kaggle.**

## Start here — no GPU retraining needed for the first experiment
1. Open this notebook in Colab. Keep `MODE = "recover"` in the first code cell.
2. Run the installation cell. If Colab asks for a restart, restart and rerun the settings cell, then continue after installation.
3. Allow Google Drive access when requested. The notebook uses your existing `/content/data` competition CSVs, or downloads them using `kaggle.json` when necessary.
4. Run all remaining cells. **Your uploaded V3 ZIP is embedded in this notebook**, so you do not need to upload it again. It contains predictions and reports, not API credentials.
5. Upload the generated **submission_v4.csv** to Kaggle. Keep the V2 submission unchanged as your current best.
6. Send **v4_results.zip** and the Kaggle score for the next analysis.

Recovery mode does CPU-based TF-IDF training and calibration; it does not train a transformer. It needs the **same train.csv, validation.csv, and test.csv** as V3. ID mismatches and changes to recorded audit text/labels stop the notebook. The archive does not contain every original text, so keep the same dataset version.

## What the V3 ZIP actually established
- The submitted predictions were **100% TF-IDF and 0% transformer**. I verified the saved probabilities reproduce the submitted CSV.
- Only **150 examples** determined final thresholds. Calibration macro F1 was **0.4878**, while the separate 150-row audit was **0.3963**.
- Audit F1 was **0.0000 threatening**, **0.1667 sexual content**, and **0.2609 sarcasm**.
- Both BanglaBERT runs reached their best recorded development score at epoch **5**, so the earlier V4's four-epoch training schedule was not supported by that result. Optional retraining here runs six epochs and compares snapshots 3–6.
- V3's TF-IDF search had a scoring error: applying `average="macro"` to a single-column binary target averages negative-class and positive-class F1. This notebook explicitly evaluates **positive-class binary F1** for each label.

The ZIP does not prove why V3 fell below V2; V2's validation probabilities and full executed report are unavailable. The new changes directly address observed weaknesses without claiming a guaranteed score.

## What this V4 changes
- Reuses both trained BanglaBERT models and their average.
- Adds three freshly fitted TF-IDF configurations so model selection is not limited to the settings chosen by V3's faulty single-label metric.
- Uses **all 500 validation labels** for final calibration, without adding them to recovery-mode model training.
- Searches real probability cutoffs, including values below **0.1**; no forced threshold of 0.5 for rare labels.
- Tests **different model/mixture choices for each label** with inner cross-validation, instead of one global transformer weight for all seven labels.
- Requires a minimum cross-validation gain and sufficient positive examples before changing a label's model. Scarce labels still receive calibrated thresholds.
- Uses outer folds to diagnose the whole model-choice + threshold-fitting procedure. A fallback compares it with simply recalibrating TF-IDF.
- Includes the known Python 3.13 installation and sample/test ID fixes.

**Rules assumed from your original notebook:** macro F1, seven binary targets, pretrained models allowed. Optional five-fold mode also assumes supplied validation labels may be used for model training, as V2 already did for TF-IDF. Verify these assumptions in the competition rules; the competition rules page could not be retrieved independently. No external labeled dataset is added.

In [ ]:
# A. Settings — run this cell first.
from pathlib import Path
MODE = "recover"                      # recommended first; "five_fold" trains new GPU models
USE_DRIVE = True
DATA_DIR = Path("/content/data")
OUTPUT_ROOT = Path("/content/drive/MyDrive/bengali_safety_v4_evidence" if USE_DRIVE else "/content/bengali_safety_v4_evidence")
SEED = 42
N_FOLDS = 5
INNER_FOLDS = 3
MIN_MODEL_SWITCH_POSITIVES = 12
MIN_MODEL_SWITCH_GAIN = 0.02           # positive-label F1 gain needed over recalibrated baseline
MIN_POLICY_GAIN = 0.005               # outer diagnostic gain needed to prefer per-label selection
BOOTSTRAPS = 8                        # stabilize final cutoffs using median resampled thresholds
REFRESH_TFIDF = True                  # CPU training; replaces dependence on V3's mistaken C search
REUSE_COMPLETED_FOLDS = True
SUBMISSION_ID_POLICY = "test"         # actual test IDs, as in your accepted V2/V3 submissions
LABELS = ["is_toxic", "implicit_toxicity", "sarcasm", "contains_profanity",
          "threatening", "identity_attack", "sexual_content"]
ID_COL, TEXT_COL = "ID", "text"
METRIC = "macro_f1"
PIPELINE_VERSION = "v4.2_evidence"

# Used only when MODE = "five_fold". Recovery mode needs no GPU.
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 192
LEARNING_RATE = 2e-5
EPOCHS = 6
SNAPSHOT_EPOCHS = [3, 4, 5, 6]
TRAIN_BATCH_SIZE = 8
ACCUMULATION_STEPS = 2
EVAL_BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
GRADIENT_CHECKPOINTING = False
SAVE_MODEL_WEIGHTS = False            # optional; enabling it saves several GB of snapshots
assert MODE in {"recover","five_fold"}
assert N_FOLDS >= 3 and INNER_FOLDS >= 2
assert SUBMISSION_ID_POLICY in {"test","strict_sample"}
print("Mode:",MODE)

In [ ]:
# B. Installation. Recovery mode does not install or train a transformer.
import sys, subprocess

def install(*packages):
    subprocess.check_call([sys.executable,"-m","pip","install",*packages])

install("iterative-stratification==0.1.9","kaggle","scikit-learn","pandas","scipy","tqdm")
install("git+https://github.com/csebuetnlp/normalizer.git")
if MODE == "five_fold":
    install("transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1","safetensors")
print("Installation complete. If asked to restart, do so; then rerun Settings and continue below.")

In [ ]:
# C. Persistent outputs.
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)

In [ ]:
# D. Get competition data. Existing files are reused; your API key is never printed or saved in reports.
import os, zipfile, shutil, subprocess
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not list(DATA_DIR.rglob("train.csv")):
    key = Path.home() / ".kaggle" / "kaggle.json"
    if not key.exists() and not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) and not os.environ.get("KAGGLE_API_TOKEN"):
        from google.colab import files
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise ValueError("Upload kaggle.json, or put the competition CSV files in DATA_DIR.")
        key.parent.mkdir(parents=True, exist_ok=True)
        key.write_bytes(uploaded["kaggle.json"])
        key.chmod(0o600)
        Path("kaggle.json").unlink(missing_ok=True)
        del uploaded
    subprocess.run(["kaggle", "competitions", "download", "-c",
                    "bengali-nlp-safety-challenge-2026", "-p", str(DATA_DIR)], check=True)
    archive = DATA_DIR / "bengali-nlp-safety-challenge-2026.zip"
    if not archive.exists():
        raise FileNotFoundError("Expected competition ZIP missing. Inspect the download output.")
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            dest = (DATA_DIR / member.filename).resolve()
            if not dest.is_relative_to(DATA_DIR.resolve()):
                raise ValueError("Unsafe ZIP path")
        z.extractall(DATA_DIR)
roots = [p.parent for p in DATA_DIR.rglob("train.csv") if (p.parent / "test.csv").exists()]
if len(roots) != 1:
    raise ValueError(f"Expected one dataset folder; found {roots}. Set DATA_DIR to the correct folder.")
DATA_DIR = roots[0]
print("Data folder:", DATA_DIR)

In [ ]:
# E. Imports, input validation, and the known sample-ID mismatch.
import gc, os, re, json, math, random, hashlib, warnings, unicodedata, zipfile, time
import importlib.metadata
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_recall_curve, precision_recall_fscore_support
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from normalizer import normalize as buet_normalize
from tqdm.auto import tqdm
from IPython.display import display

if MODE == "five_fold":
    import torch
    from torch.utils.data import Dataset, DataLoader
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    assert torch.cuda.is_available(), "Enable T4 GPU for five_fold mode. Recovery works without one."
    DEVICE=torch.device("cuda")
    def seed_everything(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic=True
        torch.backends.cudnn.benchmark=False
    seed_everything(SEED)
    print("GPU:",torch.cuda.get_device_name(0))
train_df = pd.read_csv(DATA_DIR/"train.csv", dtype={ID_COL:str})
valid_df = pd.read_csv(DATA_DIR/"validation.csv", dtype={ID_COL:str})
test_df = pd.read_csv(DATA_DIR/"test.csv", dtype={ID_COL:str})
sample = pd.read_csv(DATA_DIR/"sample_submission.csv", dtype={ID_COL:str})
for name,frame,labeled in [("train",train_df,True),("validation",valid_df,True),("test",test_df,False)]:
    required=[ID_COL,TEXT_COL]+(LABELS if labeled else [])
    assert set(required).issubset(frame.columns), f"{name}: unexpected columns {frame.columns.tolist()}"
    assert len(frame)>0 and frame[ID_COL].notna().all() and frame[ID_COL].is_unique, f"{name}: invalid IDs"
    if labeled:
        for label in LABELS: frame[label]=pd.to_numeric(frame[label],errors="raise")
        assert frame[LABELS].isin([0,1]).all().all(), f"{name}: invalid/missing labels"
    print(name, frame.shape, "| missing texts:", int(frame[TEXT_COL].isna().sum()))
assert set(sample.columns)=={ID_COL,*LABELS} and len(sample.columns)==len(LABELS)+1
assert sample[ID_COL].notna().all() and sample[ID_COL].is_unique
sample_match = len(sample)==len(test_df) and set(sample[ID_COL])==set(test_df[ID_COL])
if not sample_match:
    if SUBMISSION_ID_POLICY=="strict_sample":
        raise ValueError("Sample/test IDs differ. Verify current competition files before submitting.")
    print("KNOWN SAMPLE/TEST MISMATCH: retaining actual test.csv IDs, as in your accepted submissions.")
    print("Only-in-test IDs:", len(set(test_df[ID_COL])-set(sample[ID_COL])))
    print("Only-in-sample IDs:", len(set(sample[ID_COL])-set(test_df[ID_COL])))
submission_columns=sample.columns.tolist()
submission_ids=sample[ID_COL].tolist() if sample_match else test_df[ID_COL].tolist()

In [ ]:
# F. Your supplied V3 predictions, bundled for reuse. No credentials are included.
import base64, io
V3_ZIP_SHA256 = '97263d91c5bf7297b83926386fb2c6c9cc9ca7c564ed88d3537ab658dcc167fc'
V3_ZIP_BASE64 = '...'
V3_BYTES = base64.b64decode(V3_ZIP_BASE64)
assert hashlib.sha256(V3_BYTES).hexdigest() == V3_ZIP_SHA256
print("Bundled V3 results loaded.")

In [ ]:
# G. Shared helpers. Every single-label score below uses binary positive-class F1.
def clean_text(value):
    if pd.isna(value): return ""
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC",str(value))).strip()

def model_text(value):
    value=clean_text(value)
    return buet_normalize(value) if value else ""

def make_group_folds(frame,n_folds,seed):
    # OR labels within each group are used only for allocating whole groups to folds.
    group_labels=frame.groupby("group",sort=True)[LABELS].max()
    assert len(group_labels)>=n_folds, "Too few independent text groups."
    splitter=MultilabelStratifiedKFold(n_splits=n_folds,shuffle=True,random_state=seed)
    mapping={}
    for fold,(_,holdout_groups) in enumerate(splitter.split(np.zeros((len(group_labels),1)),group_labels.to_numpy())):
        for group in group_labels.index[holdout_groups]: mapping[group]=fold
    assignment=frame["group"].map(mapping).to_numpy(dtype=int)
    assert set(assignment)==set(range(n_folds))
    for fold in range(n_folds):
        left=set(frame.loc[assignment!=fold,"group"])
        right=set(frame.loc[assignment==fold,"group"])
        assert left.isdisjoint(right)
    return assignment

def check_probabilities(p,shape):
    p=np.asarray(p)
    assert p.shape==tuple(shape), f"Wrong probability shape: {p.shape}, expected {shape}"
    assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all(), "Invalid probabilities"
    return p

def best_binary_threshold(target,probability):
    target=np.asarray(target,dtype=int)
    probability=np.asarray(probability,dtype=float)
    if target.sum()==0: return 1.000001    # no positive evidence; predict none
    if target.sum()==len(target): return 0.0
    precision,recall,thresholds=precision_recall_curve(target,probability)
    scores=2*precision[:-1]*recall[:-1]/np.maximum(precision[:-1]+recall[:-1],1e-12)
    tied=np.flatnonzero(np.isclose(scores,scores.max(),atol=1e-12,rtol=0))
    # Deterministic tie-breaking; prefer the highest threshold among equal F1 values.
    return float(thresholds[tied[-1]])

def fit_thresholds(target,probability,bootstraps=BOOTSTRAPS,seed=SEED):
    target=np.asarray(target)
    probability=check_probabilities(probability,target.shape)
    rng=np.random.default_rng(seed)
    estimates=[np.array([best_binary_threshold(target[:,j],probability[:,j]) for j in range(target.shape[1])])]
    # Resample whole rows to retain within-row label correlations.
    for _ in range(bootstraps):
        idx=rng.integers(0,len(target),len(target))
        estimates.append(np.array([best_binary_threshold(target[idx,j],probability[idx,j]) for j in range(target.shape[1])]))
    return np.median(np.stack(estimates),axis=0)

def macro_score(target,prediction):
    return float(f1_score(target,prediction,average="macro",zero_division=0))

def positive_f1(target,prediction):
    return float(f1_score(np.asarray(target).reshape(-1),np.asarray(prediction).reshape(-1),average="binary",pos_label=1,zero_division=0))

## Recovery: reuse the models you have already trained
The embedded archive has validation/test predictions for TF-IDF and both BanglaBERT runs. We align predictions with the CSV labels by **ID**, verify the saved audit labels against the current CSV, and reconstruct V3's filtered training rows before refreshing TF-IDF. A mismatch stops the run.

The following recovery step trains only TF-IDF models on CPU. The optional GPU section after it is skipped automatically in recovery mode.

In [ ]:
# H. Load, verify, and augment cached V3 predictions.
def load_v3_archive(blob,validation,test,expected_labels):
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        report=json.loads(archive.read("experiment_report.json"))
        assert report["identity"]["labels"]==expected_labels
        def arrays(name):
            with np.load(io.BytesIO(archive.read(name)),allow_pickle=False) as z:
                return {k:z[k].copy() for k in z.files}
        selected=arrays("selected_predictions.npz")
        saved_valid_ids=selected["valid_ids"].astype(str)
        saved_test_ids=selected["test_ids"].astype(str)
        assert len(set(saved_valid_ids))==len(saved_valid_ids) and len(set(saved_test_ids))==len(saved_test_ids)
        assert len(validation)==len(saved_valid_ids) and set(validation[ID_COL])==set(saved_valid_ids), "Validation IDs differ from saved V3. Use the same dataset version."
        assert len(test)==len(saved_test_ids) and set(test[ID_COL])==set(saved_test_ids), "Test IDs differ from saved V3. Use the same dataset version."
        vp=pd.Index(saved_valid_ids).get_indexer(validation[ID_COL])
        tp=pd.Index(saved_test_ids).get_indexer(test[ID_COL])
        linear=arrays("tfidf_predictions.npz")
        names=["v3_tfidf"]
        valid_probs=[linear["valid"][vp]]
        test_probs=[linear["test"][tp]]
        transformer_v,transformer_t=[],[]
        for name in sorted(archive.namelist()):
            if name.endswith("/predictions.npz"):
                item=arrays(name)
                model_name=name.split("/")[0]
                names.append(model_name)
                valid_probs.append(item["valid"][vp])
                test_probs.append(item["test"][tp])
                transformer_v.append(item["valid"][vp])
                transformer_t.append(item["test"][tp])
        assert len(transformer_v)==2, "Expected the two supplied V3 transformer runs."
        names.append("v3_transformer_mean")
        valid_probs.append(np.mean(transformer_v,axis=0))
        test_probs.append(np.mean(transformer_t,axis=0))
        # Validate all 150 original audit labels: wrong rows are explicit, correct rows
        # equal the saved selected predictions. This is a consistency check, not new labels.
        split=pd.read_csv(io.BytesIO(archive.read("validation_split.csv")),dtype={ID_COL:str})
        errors=pd.read_csv(io.BytesIO(archive.read("audit_errors.csv")),dtype={ID_COL:str})
        assert split[ID_COL].is_unique and set(split[ID_COL])==set(saved_valid_ids)
        predicted=(selected["valid"]>=selected["thresholds"]).astype(int)
        lookup=pd.DataFrame(predicted,index=saved_valid_ids,columns=expected_labels)
        audit_ids=split.loc[split.role=="audit",ID_COL].tolist()
        known=lookup.loc[audit_ids].copy()
        for _,row in errors.iterrows():
            assert row[ID_COL] in known.index
            known.loc[row[ID_COL]]=[row[label+"_true"] for label in expected_labels]
        actual=validation.set_index(ID_COL).loc[audit_ids,expected_labels]
        assert np.array_equal(actual.to_numpy(),known.to_numpy()), "Validation annotations differ from your saved V3 audit."
        current_text=validation.set_index(ID_COL)[TEXT_COL].map(clean_text)
        assert all(current_text.loc[row[ID_COL]]==clean_text(row[TEXT_COL]) for _,row in errors.iterrows()), "Validation text differs from saved V3."
        for p in valid_probs:check_probabilities(p,(len(validation),len(expected_labels)))
        for p in test_probs:check_probabilities(p,(len(test),len(expected_labels)))
        return names,valid_probs,test_probs,report

if MODE == "recover":
    for frame in [train_df,valid_df,test_df]:
        frame["clean_text"]=frame[TEXT_COL].map(clean_text)
        frame["model_text"]=frame.clean_text.map(model_text)
        frame["group"]=frame.model_text.map(clean_text)
    base_names,base_valid,base_test,v3_report=load_v3_archive(V3_BYTES,valid_df,test_df,LABELS)
    meta_frame=valid_df.copy().reset_index(drop=True)
    y_meta=meta_frame[LABELS].to_numpy(dtype=np.int8)
    # Same exclusion and deduplication policy as V3; models never train on validation.
    overlap=train_df.group.isin(valid_df.group)|train_df[ID_COL].isin(valid_df[ID_COL])
    fit_frame=train_df.loc[~overlap].drop_duplicates(["group"]+LABELS).reset_index(drop=True)
    assert len(fit_frame)==v3_report["data_rows"]["train"], "Training rows differ from V3. Inspect data/normalizer versions."
    assert set(fit_frame.group).isdisjoint(valid_df.group)
    assert set(fit_frame[ID_COL]).isdisjoint(test_df[ID_COL])
    signature=hashlib.sha256(V3_BYTES)
    signature.update(json.dumps({"version":PIPELINE_VERSION,"seed":SEED,"refresh":REFRESH_TFIDF,
        "sklearn":importlib.metadata.version("scikit-learn")},sort_keys=True).encode())
    signature.update(fit_frame[[ID_COL,"clean_text"]+LABELS].to_csv(index=False).encode())
    signature.update(valid_df[[ID_COL,"clean_text"]+LABELS].to_csv(index=False).encode())
    signature.update(test_df[[ID_COL,"clean_text"]].to_csv(index=False).encode())
    RUN_DIR=OUTPUT_ROOT/("recover_"+signature.hexdigest()[:16])
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    print("Verified validation rows:",len(meta_frame),"| verified V3 training rows:",len(fit_frame))
    if REFRESH_TFIDF:
        refresh_configs=[("fresh_balanced_C2",2.0,"balanced"),
                         ("fresh_balanced_C8",8.0,"balanced"),("fresh_plain_C8",8.0,None)]
        cache=RUN_DIR/"fresh_tfidf.npz"
        if cache.exists() and REUSE_COMPLETED_FOLDS:
            with np.load(cache,allow_pickle=False) as z:
                for name,_,_ in refresh_configs:
                    base_names.append(name)
                    base_valid.append(check_probabilities(z[name+"_valid"].copy(),y_meta.shape))
                    base_test.append(check_probabilities(z[name+"_test"].copy(),(len(test_df),len(LABELS))))
            print("Reused completed CPU TF-IDF models' predictions.")
        else:
            word=TfidfVectorizer(tokenizer=str.split,token_pattern=None,lowercase=False,ngram_range=(1,2),
                min_df=2,max_features=120_000,sublinear_tf=True,dtype=np.float32)
            char=TfidfVectorizer(analyzer="char_wb",lowercase=False,ngram_range=(3,6),
                min_df=2,max_features=180_000,sublinear_tf=True,dtype=np.float32)
            X=hstack([word.fit_transform(fit_frame.clean_text),char.fit_transform(fit_frame.clean_text)],format="csr")
            V=hstack([word.transform(valid_df.clean_text),char.transform(valid_df.clean_text)],format="csr")
            T=hstack([word.transform(test_df.clean_text),char.transform(test_df.clean_text)],format="csr")
            payload={}
            for name,c,weight in refresh_configs:
                print("CPU training:",name)
                pv=np.zeros(y_meta.shape,dtype=np.float32)
                pt=np.zeros((len(test_df),len(LABELS)),dtype=np.float32)
                for j,label in enumerate(LABELS):
                    target=fit_frame[label].to_numpy()
                    if len(np.unique(target))<2:
                        pv[:,j]=pt[:,j]=float(target.mean())
                    else:
                        clf=LogisticRegression(C=c,class_weight=weight,solver="liblinear",max_iter=2500,random_state=SEED)
                        clf.fit(X,target)
                        pv[:,j]=clf.predict_proba(V)[:,1]
                        pt[:,j]=clf.predict_proba(T)[:,1]
                base_names.append(name);base_valid.append(pv);base_test.append(pt)
                payload[name+"_valid"]=pv;payload[name+"_test"]=pt
            temp=RUN_DIR/"fresh_tfidf.tmp.npz"
            np.savez_compressed(temp,**payload);temp.replace(cache)
            del X,V,T,word,char
            gc.collect()
    base_valid=np.stack(base_valid,axis=1)
    base_test=np.stack(base_test,axis=1)
    outer_folds=make_group_folds(meta_frame,N_FOLDS,SEED)
    source_report={"mode":MODE,"v3_zip_sha256":V3_ZIP_SHA256,"v3_report":v3_report,
                   "fresh_tfidf":REFRESH_TFIDF,"training_rows":len(fit_frame)}
else:
    print("Recovery step skipped; five-fold mode trains new models below.")

## Optional five-fold retraining — skipped in recovery mode
Change `MODE` to `"five_fold"` only when you want a separate, more expensive experiment. Enable a T4 GPU, then rerun from Settings. This trains **five BanglaBERT models for six epochs each**, pools train + supplied labeled validation, and compares snapshots from epochs 3, 4, 5, and 6 plus the mean of 5–6. It can take several hours.

It does not blend these OOF predictions with old V3 validation predictions, because their rows and training histories differ. Completed fold caches are reused. Partial folds restart from epoch 1. `SAVE_MODEL_WEIGHTS=False` saves probabilities and reports, not reusable model weights.

In [ ]:
if MODE == "five_fold":
    # 5. Preserve text, combine permitted labeled data, and create grouped multilabel folds.
    def clean_text(value):
        if pd.isna(value): return ""
        return re.sub(r"\s+", " ", unicodedata.normalize("NFC",str(value))).strip()

    def model_text(value):
        value=clean_text(value)
        return buet_normalize(value) if value else ""

    for frame in [train_df,valid_df,test_df]:
        frame["clean_text"]=frame[TEXT_COL].map(clean_text)
        frame["model_text"]=frame["clean_text"].map(model_text)
        frame["group"]=frame["model_text"].map(clean_text)
    train_df["source"]="train"
    valid_df["source"]="validation"
    labeled=pd.concat([train_df,valid_df],ignore_index=True)
    # A repeated ID is safe to consolidate only if text and targets agree.
    repeated=labeled[labeled[ID_COL].duplicated(keep=False)]
    if len(repeated):
        same=repeated.groupby(ID_COL)[["clean_text"]+LABELS].nunique(dropna=False).le(1).all().all()
        assert same, "Repeated IDs have conflicting content across train/validation. Check dataset versions."
        labeled=labeled.drop_duplicates(ID_COL).reset_index(drop=True)
    assert set(labeled[ID_COL]).isdisjoint(test_df[ID_COL]), "Labeled/test IDs overlap; verify the dataset version."
    y=labeled[LABELS].to_numpy(dtype=np.int8)
    conflicts=labeled.groupby("group")[LABELS].nunique().gt(1).any(axis=1)
    print("Conflicting-label text groups (retained):",int(conflicts.sum()))
    print("Extra rows sharing normalized text (retained, grouped):",int(labeled.group.duplicated().sum()))
    print("Normalized blank texts:",int(labeled.group.eq("").sum()))
    print("Labeled/test shared text groups (reported only):",len(set(labeled.group)&set(test_df.group)))

    def make_group_folds(frame,n_folds,seed):
        # OR labels within each group are used only for allocating whole groups to folds.
        group_labels=frame.groupby("group",sort=True)[LABELS].max()
        assert len(group_labels)>=n_folds, "Too few independent text groups."
        splitter=MultilabelStratifiedKFold(n_splits=n_folds,shuffle=True,random_state=seed)
        mapping={}
        for fold,(_,holdout_groups) in enumerate(splitter.split(np.zeros((len(group_labels),1)),group_labels.to_numpy())):
            for group in group_labels.index[holdout_groups]: mapping[group]=fold
        assignment=frame["group"].map(mapping).to_numpy(dtype=int)
        assert set(assignment)==set(range(n_folds))
        for fold in range(n_folds):
            left=set(frame.loc[assignment!=fold,"group"])
            right=set(frame.loc[assignment==fold,"group"])
            assert left.isdisjoint(right)
        return assignment

    fold_ids=make_group_folds(labeled,N_FOLDS,SEED)
    fold_support=pd.DataFrame({f"fold_{f}":y[fold_ids==f].sum(0) for f in range(N_FOLDS)},index=LABELS)
    fold_support["total"]=y.sum(0)
    display(fold_support)
    print("Fold row counts:",np.bincount(fold_ids).tolist())
    if (fold_support.drop(columns="total")<5).any().any():
        warnings.warn("Some folds have fewer than five positives for a label. OOF pooling helps, but these estimates remain noisy.")
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
if MODE == "five_fold":
    # 6. Identity and cache checks prevent reusing old V3 or mismatched V4 predictions.
    versions={p:importlib.metadata.version(p) for p in ["torch","transformers","tokenizers","scikit-learn","numpy","iterative-stratification"]}
    settings={"pipeline":PIPELINE_VERSION,"model":MODEL_NAME,"folds":N_FOLDS,"seed":SEED,
        "length":MAX_LENGTH,"lr":LEARNING_RATE,"epochs":EPOCHS,"snapshots":SNAPSHOT_EPOCHS,
        "batch":TRAIN_BATCH_SIZE,"accumulation":ACCUMULATION_STEPS,"weight_decay":WEIGHT_DECAY,
        "warmup":WARMUP_RATIO,"gradient_checkpointing":GRADIENT_CHECKPOINTING,
        "labels":LABELS,"versions":versions,"save_weights":SAVE_MODEL_WEIGHTS}
    fingerprint=hashlib.sha256(json.dumps(settings,sort_keys=True).encode())
    for frame in [labeled,test_df]:
        columns=[ID_COL,"clean_text","model_text"]+(LABELS if set(LABELS).issubset(frame.columns) else [])
        fingerprint.update(frame[columns].to_csv(index=False).encode())
    fingerprint.update(np.asarray(fold_ids,dtype=np.int64).tobytes())
    RUN_DIR=OUTPUT_ROOT/fingerprint.hexdigest()[:16]
    RUN_DIR.mkdir(parents=True,exist_ok=True)
    labeled[[ID_COL,"source"]].assign(fold=fold_ids).to_csv(RUN_DIR/"fold_assignment.csv",index=False)
    fold_support.to_csv(RUN_DIR/"fold_support.csv")
    print("Results folder:",RUN_DIR)

    def check_probabilities(p,shape):
        p=np.asarray(p)
        assert p.shape==tuple(shape), f"Wrong probability shape: {p.shape}, expected {shape}"
        assert np.isfinite(p).all() and ((p>=0)&(p<=1)).all(), "Invalid probabilities"
        return p
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
if MODE == "five_fold":
    # 8. Fold-wise TF-IDF baselines; vocabulary is fit on each training fold only.
    LINEAR_CONFIGS=[{"name":"tfidf_balanced_C2","C":2.0,"class_weight":"balanced"},
                    {"name":"tfidf_plain_C4","C":4.0,"class_weight":None}]

    def linear_fold(train_index,holdout_index):
        word=TfidfVectorizer(tokenizer=str.split,token_pattern=None,lowercase=False,
            ngram_range=(1,2),min_df=2,max_features=120_000,sublinear_tf=True,dtype=np.float32)
        char=TfidfVectorizer(analyzer="char_wb",lowercase=False,ngram_range=(3,6),
            min_df=2,max_features=180_000,sublinear_tf=True,dtype=np.float32)
        train_text=labeled.iloc[train_index].clean_text
        X=hstack([word.fit_transform(train_text),char.fit_transform(train_text)],format="csr")
        H=hstack([word.transform(labeled.iloc[holdout_index].clean_text),char.transform(labeled.iloc[holdout_index].clean_text)],format="csr")
        T=hstack([word.transform(test_df.clean_text),char.transform(test_df.clean_text)],format="csr")
        output={}
        for cfg in LINEAR_CONFIGS:
            hold_prob=np.zeros((len(holdout_index),len(LABELS)),dtype=np.float32)
            test_prob=np.zeros((len(test_df),len(LABELS)),dtype=np.float32)
            for j,label in enumerate(LABELS):
                target=y[train_index,j]
                if len(np.unique(target))<2:
                    hold_prob[:,j]=test_prob[:,j]=float(target.mean())
                    continue
                clf=LogisticRegression(C=cfg["C"],class_weight=cfg["class_weight"],
                    solver="liblinear",max_iter=2500,random_state=SEED)
                clf.fit(X,target)
                hold_prob[:,j]=clf.predict_proba(H)[:,1]
                test_prob[:,j]=clf.predict_proba(T)[:,1]
            output[cfg["name"]]=(hold_prob,test_prob)
        return output

    linear_oof={c["name"]:np.full(y.shape,np.nan,dtype=np.float32) for c in LINEAR_CONFIGS}
    linear_test_parts={c["name"]:[] for c in LINEAR_CONFIGS}
    for fold in range(N_FOLDS):
        train_index=np.flatnonzero(fold_ids!=fold)
        holdout_index=np.flatnonzero(fold_ids==fold)
        folder=RUN_DIR/f"fold_{fold}"
        folder.mkdir(exist_ok=True)
        cache=folder/"linear.npz"
        if REUSE_COMPLETED_FOLDS and cache.exists():
            with np.load(cache,allow_pickle=False) as z:
                assert np.array_equal(z["holdout_index"],holdout_index)
                output={c["name"]:(z[c["name"]+"_hold"].copy(),z[c["name"]+"_test"].copy()) for c in LINEAR_CONFIGS}
        else:
            print("TF-IDF fold",fold+1,"/",N_FOLDS)
            output=linear_fold(train_index,holdout_index)
            payload={"holdout_index":holdout_index}
            for name,(hp,tp) in output.items():
                payload[name+"_hold"],payload[name+"_test"]=hp,tp
            temp=folder/"linear.tmp.npz"
            np.savez_compressed(temp,**payload)
            temp.replace(cache)
        for name,(hp,tp) in output.items():
            linear_oof[name][holdout_index]=check_probabilities(hp,(len(holdout_index),len(LABELS)))
            linear_test_parts[name].append(check_probabilities(tp,(len(test_df),len(LABELS))))
        gc.collect()
    linear_test={name:np.mean(parts,axis=0) for name,parts in linear_test_parts.items()}
    for name in linear_oof: check_probabilities(linear_oof[name],y.shape)
    print("TF-IDF OOF predictions complete.")
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
if MODE == "five_fold":
    # 9. Tokenize once, use dynamic padding, and keep labels as float32.
    class EncodedDataset(Dataset):
        def __init__(self,encodings,indices,labels=None):
            self.encodings=encodings
            self.indices=np.asarray(indices)
            self.labels=labels
        def __len__(self): return len(self.indices)
        def __getitem__(self,index):
            row=self.indices[index]
            item={k:v[row] for k,v in self.encodings.items()}
            if self.labels is not None: item["labels"]=self.labels[row].astype(np.float32)
            return item

    class LabelCollator:
        def __init__(self,tokenizer): self.tokenizer=tokenizer
        def __call__(self,items):
            inputs=[{k:v for k,v in item.items() if k!="labels"} for item in items]
            batch=self.tokenizer.pad(inputs,padding=True,pad_to_multiple_of=8,return_tensors="pt")
            if "labels" in items[0]:
                batch["labels"]=torch.tensor(np.stack([i["labels"] for i in items]),dtype=torch.float32)
            return batch

    tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
    probe=labeled.model_text.sample(min(2000,len(labeled)),random_state=SEED).tolist()
    lengths=np.array([len(x) for x in tokenizer(probe,truncation=False)["input_ids"]])
    print("Token length p50/p90/p95/p99:",np.percentile(lengths,[50,90,95,99]))
    print("Fraction longer than MAX_LENGTH:",float((lengths>MAX_LENGTH).mean()))
    encoded_labeled=tokenizer(labeled.model_text.tolist(),truncation=True,max_length=MAX_LENGTH,padding=False)
    encoded_test=tokenizer(test_df.model_text.tolist(),truncation=True,max_length=MAX_LENGTH,padding=False)
    collator=LabelCollator(tokenizer)

    def make_loader(encodings,indices,labels=None,training=False):
        dataset=EncodedDataset(encodings,indices,labels)
        return DataLoader(dataset,batch_size=TRAIN_BATCH_SIZE if training else EVAL_BATCH_SIZE,
            shuffle=training,num_workers=0,pin_memory=True,collate_fn=collator)

    @torch.no_grad()
    def predict(model,loader):
        model.eval()
        chunks=[]
        for batch in loader:
            inputs={k:v.to(DEVICE) for k,v in batch.items() if k!="labels"}
            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inputs).logits
            chunks.append(torch.sigmoid(logits.float()).cpu().numpy())
        return check_probabilities(np.concatenate(chunks),(len(loader.dataset),len(LABELS)))
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
if MODE == "five_fold":
    # 10. Train one fold at a time. Fixed training duration avoids per-fold validation early stopping.
    def train_transformer_fold(fold):
        train_index=np.flatnonzero(fold_ids!=fold)
        holdout_index=np.flatnonzero(fold_ids==fold)
        folder=RUN_DIR/f"fold_{fold}"
        folder.mkdir(exist_ok=True)
        cache,metadata_path=folder/"transformer.npz",folder/"transformer.json"
        if REUSE_COMPLETED_FOLDS and cache.exists() and metadata_path.exists():
            metadata=json.loads(metadata_path.read_text())
            assert metadata["settings"]==settings and metadata["snapshot_epochs"]==SNAPSHOT_EPOCHS
            with np.load(cache,allow_pickle=False) as z:
                assert np.array_equal(z["holdout_index"],holdout_index)
                hp,tp=z["hold"].copy(),z["test"].copy()
            check_probabilities(hp,(len(SNAPSHOT_EPOCHS),len(holdout_index),len(LABELS)))
            check_probabilities(tp,(len(SNAPSHOT_EPOCHS),len(test_df),len(LABELS)))
            print("Reusing completed transformer fold",fold+1)
            return holdout_index,hp,tp,metadata
        seed_everything(SEED+fold)
        train_loader=make_loader(encoded_labeled,train_index,y,training=True)
        hold_loader=make_loader(encoded_labeled,holdout_index)
        test_loader=make_loader(encoded_test,np.arange(len(test_df)))
        model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=len(LABELS),
            problem_type="multi_label_classification",id2label=dict(enumerate(LABELS)),
            label2id={label:i for i,label in enumerate(LABELS)}).to(DEVICE)
        if GRADIENT_CHECKPOINTING: model.gradient_checkpointing_enable()
        groups=[{"params":[p for p in model.parameters() if p.requires_grad and p.ndim>1],"weight_decay":WEIGHT_DECAY},
                {"params":[p for p in model.parameters() if p.requires_grad and p.ndim<=1],"weight_decay":0.0}]
        optimizer=torch.optim.AdamW(groups,lr=LEARNING_RATE)
        steps=math.ceil(len(train_loader)/ACCUMULATION_STEPS)*EPOCHS
        scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*WARMUP_RATIO),steps)
        scaler=torch.amp.GradScaler("cuda")
        loss_fn=torch.nn.BCEWithLogitsLoss()
        history=[]
        hold_snapshots,test_snapshots={},{}
        started=time.time()
        for epoch in range(1,EPOCHS+1):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            total_loss=0.
            for step,batch in enumerate(tqdm(train_loader,desc=f"Fold {fold+1}/{N_FOLDS}, epoch {epoch}/{EPOCHS}")):
                target=batch.pop("labels").to(DEVICE)
                inputs={k:v.to(DEVICE) for k,v in batch.items()}
                group_start=(step//ACCUMULATION_STEPS)*ACCUMULATION_STEPS
                group_samples=min(TRAIN_BATCH_SIZE*ACCUMULATION_STEPS,len(train_index)-group_start*TRAIN_BATCH_SIZE)
                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inputs).logits
                    loss=loss_fn(logits,target)
                if not torch.isfinite(loss): raise FloatingPointError("Non-finite training loss.")
                scaler.scale(loss*(len(target)/group_samples)).backward()
                total_loss+=loss.item()*len(target)
                if (step+1)%ACCUMULATION_STEPS==0 or step+1==len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    old_scale=scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    if scaler.get_scale()>=old_scale: scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
            row={"fold":fold,"epoch":epoch,"train_loss":total_loss/len(train_index),"elapsed_seconds":time.time()-started}
            if epoch in SNAPSHOT_EPOCHS:
                hp=predict(model,hold_loader)
                tp=predict(model,test_loader)
                hold_snapshots[epoch],test_snapshots[epoch]=hp,tp
                # Display only; this does not choose the checkpoint or alter training duration.
                row["holdout_macro_f1_at_0_5"]=macro_score(y[holdout_index],hp>=.5)
                if SAVE_MODEL_WEIGHTS:
                    model.save_pretrained(folder/f"epoch_{epoch}",safe_serialization=True)
                    tokenizer.save_pretrained(folder/f"epoch_{epoch}")
            history.append(row)
            pd.DataFrame(history).to_csv(folder/"history.csv",index=False)
            print(row)
        hp=np.stack([hold_snapshots[e] for e in SNAPSHOT_EPOCHS])
        tp=np.stack([test_snapshots[e] for e in SNAPSHOT_EPOCHS])
        metadata={"settings":settings,"fold":fold,"snapshot_epochs":SNAPSHOT_EPOCHS,"history":history,
                  "resolved_model_commit":getattr(model.config,"_commit_hash",None)}
        temp=folder/"transformer.tmp.npz"
        np.savez_compressed(temp,holdout_index=holdout_index,hold=hp,test=tp)
        temp.replace(cache)
        metadata_path.write_text(json.dumps(metadata,indent=2))
        del model,optimizer,scheduler,scaler,groups,train_loader,hold_loader,test_loader
        gc.collect(); torch.cuda.empty_cache()
        return holdout_index,hp,tp,metadata
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
if MODE == "five_fold":
    # 11. Generate OOF predictions for every row; average test predictions across ALL folds.
    transformer_oof=np.full((len(SNAPSHOT_EPOCHS),len(labeled),len(LABELS)),np.nan,dtype=np.float32)
    transformer_test_parts=[]
    run_metadata=[]
    coverage=np.zeros(len(labeled),dtype=int)
    for fold in range(N_FOLDS):
        index,hp,tp,metadata=train_transformer_fold(fold)
        assert not coverage[index].any(), "A row was predicted as holdout more than once."
        transformer_oof[:,index,:]=hp
        transformer_test_parts.append(tp)
        coverage[index]+=1
        run_metadata.append(metadata)
    assert (coverage==1).all()
    check_probabilities(transformer_oof,(len(SNAPSHOT_EPOCHS),len(labeled),len(LABELS)))
    transformer_test=np.mean(transformer_test_parts,axis=0)
    check_probabilities(transformer_test,(len(SNAPSHOT_EPOCHS),len(test_df),len(LABELS)))
    np.savez_compressed(RUN_DIR/"transformer_all_predictions.npz",oof=transformer_oof,test=transformer_test,
        epochs=np.asarray(SNAPSHOT_EPOCHS),folds=fold_ids,
        labeled_ids=labeled[ID_COL].to_numpy(dtype=str),test_ids=test_df[ID_COL].to_numpy(dtype=str))
    print("Every labeled row has an OOF prediction. Every fold contributes to test predictions.")
else:
    print("Five-fold training step skipped: recovery mode uses saved models.")

In [ ]:
# I. Prepare the same selection interface for newly trained OOF models.
if MODE == "five_fold":
    base_names=list(linear_oof)
    base_valid=[linear_oof[name] for name in base_names]
    base_test=[linear_test[name] for name in base_names]
    for epoch in SNAPSHOT_EPOCHS:
        index=SNAPSHOT_EPOCHS.index(epoch)
        base_names.append(f"new_banglabert_epoch_{epoch}")
        base_valid.append(transformer_oof[index])
        base_test.append(transformer_test[index])
    indices=[SNAPSHOT_EPOCHS.index(e) for e in [5,6]]
    base_names.append("new_banglabert_mean_5_6")
    base_valid.append(transformer_oof[indices].mean(0))
    base_test.append(transformer_test[indices].mean(0))
    base_valid=np.stack(base_valid,axis=1)
    base_test=np.stack(base_test,axis=1)
    meta_frame=labeled.copy().reset_index(drop=True)
    y_meta=y.copy()
    outer_folds=fold_ids.copy()
    source_report={"mode":MODE,"settings":settings,"runs":run_metadata}
print("Available models:",base_names)
print("Calibration/selection rows:",len(y_meta))

## Selection: learn which model helps each label
We compare each individual model and a small set of mixtures between the baseline TF-IDF and each transformer. For each label, inner folds fit thresholds on one subset and measure **positive-class F1** on another. A model switch requires enough positive examples and a predeclared improvement over the baseline. The threshold is then fitted again on all available calibration rows for that fit.

Outer folds evaluate this entire selection procedure. We also evaluate a simpler policy that only recalibrates baseline TF-IDF. If the per-label policy fails to improve the outer diagnostic by the configured minimum, the notebook uses the simpler policy.

**These scores are development diagnostics, not unbiased Kaggle predictions.** V3 already used some of these validation labels to choose checkpoints and TF-IDF settings; the audit results have now also influenced our design. Five-fold base models share training data across folds. Choosing between policies using the outer result introduces further selection optimism. The final all-data fitted score is optimistic as well. Keep public leaderboard tests few and deliberate.

In [ ]:
# J. Candidate probabilities, with identical weights for validation and test.
def make_candidates(names,pvalid,ptest):
    count=len(names)
    weights=[np.eye(count)[i] for i in range(count)]
    candidate_names=list(names)
    for i,name in enumerate(names):
        if i==0 or not ("banglabert" in name or "transformer_mean" in name):continue
        for transformer_weight in [.25,.50,.75]:
            w=np.zeros(count);w[0]=1-transformer_weight;w[i]=transformer_weight
            weights.append(w)
            candidate_names.append(f"baseline + {transformer_weight:.2f} {name}")
    weights=np.stack(weights)
    valid=np.einsum("nml,cm->ncl",pvalid,weights)
    test=np.einsum("nml,cm->ncl",ptest,weights)
    check_probabilities(valid,(pvalid.shape[0],len(weights),pvalid.shape[2]))
    check_probabilities(test,(ptest.shape[0],len(weights),ptest.shape[2]))
    return candidate_names,weights,valid,test

candidate_names,candidate_weights,candidate_valid,candidate_test=make_candidates(base_names,base_valid,base_test)
print("Candidate count:",len(candidate_names))
meta_frame[[ID_COL]].assign(fold=outer_folds).to_csv(RUN_DIR/"selection_folds.csv",index=False)
support=pd.DataFrame({f"fold_{f}":y_meta[outer_folds==f].sum(0) for f in range(N_FOLDS)},index=LABELS)
support["total"]=y_meta.sum(0)
display(support)
if (support.drop(columns="total")<3).any().any():
    warnings.warn("Very small positive counts remain. Rare-label improvements are uncertain even with cross-validation.")

In [ ]:
# K. Correct binary F1, inner model selection, and outer evaluation.
def select_policy(indices):
    target=y_meta[indices]
    probs=candidate_valid[indices]
    inner=make_group_folds(meta_frame.iloc[indices].reset_index(drop=True),INNER_FOLDS,SEED+11)
    scores=np.zeros((len(candidate_names),len(LABELS)))
    for candidate in range(len(candidate_names)):
        predictions=np.zeros(target.shape,dtype=np.int8)
        for fold in range(INNER_FOLDS):
            fit=inner!=fold;held=inner==fold
            # Exact threshold search inside inner CV; no held-out label enters this fit.
            thresholds=fit_thresholds(target[fit],probs[fit,candidate,:],bootstraps=0)
            predictions[held]=(probs[held,candidate,:]>=thresholds).astype(np.int8)
        for j in range(len(LABELS)):
            scores[candidate,j]=positive_f1(target[:,j],predictions[:,j])
    chosen=np.zeros(len(LABELS),dtype=int)
    for j in range(len(LABELS)):
        winner=int(np.argmax(scores[:,j]))
        enough=min(int(target[:,j].sum()),int(len(target)-target[:,j].sum()))>=MIN_MODEL_SWITCH_POSITIVES
        if enough and scores[winner,j]>=scores[0,j]+MIN_MODEL_SWITCH_GAIN:
            chosen[j]=winner
    chosen_prob=np.column_stack([probs[:,chosen[j],j] for j in range(len(LABELS))])
    thresholds=fit_thresholds(target,chosen_prob)
    return chosen,thresholds,scores

policy_predictions=np.zeros(y_meta.shape,dtype=np.int8)
baseline_predictions=np.zeros(y_meta.shape,dtype=np.int8)
outer_choices=[]
for fold in tqdm(range(N_FOLDS),desc="Outer evaluation of model/threshold selection"):
    fitting=np.flatnonzero(outer_folds!=fold)
    held=np.flatnonzero(outer_folds==fold)
    chosen,thresholds,scores=select_policy(fitting)
    held_prob=np.column_stack([candidate_valid[held,chosen[j],j] for j in range(len(LABELS))])
    policy_predictions[held]=(held_prob>=thresholds).astype(np.int8)
    baseline_thresholds=fit_thresholds(y_meta[fitting],candidate_valid[fitting,0,:])
    baseline_predictions[held]=(candidate_valid[held,0,:]>=baseline_thresholds).astype(np.int8)
    outer_choices.append({"fold":fold,"candidates":[candidate_names[i] for i in chosen],
                          "thresholds":thresholds.tolist(),"baseline_thresholds":baseline_thresholds.tolist()})
policy_score=macro_score(y_meta,policy_predictions)
baseline_score=macro_score(y_meta,baseline_predictions)
use_per_label=policy_score>=baseline_score+MIN_POLICY_GAIN
print("Recalibrated baseline TF-IDF outer diagnostic:",round(baseline_score,4))
print("Per-label selection outer diagnostic:",round(policy_score,4))
print("Final policy:","per-label selection" if use_per_label else "recalibrated baseline TF-IDF")
print("These are development scores; V2's 0.516 is a Kaggle score on different data.")

# Refit the selected calibration policy on all available rows, never on test labels.
chosen,final_thresholds,final_inner_scores=select_policy(np.arange(len(y_meta)))
if not use_per_label:
    chosen=np.zeros(len(LABELS),dtype=int)
    final_thresholds=fit_thresholds(y_meta,candidate_valid[:,0,:])
final_valid=np.column_stack([candidate_valid[:,chosen[j],j] for j in range(len(LABELS))])
final_test=np.column_stack([candidate_test[:,chosen[j],j] for j in range(len(LABELS))])
selected_outer_predictions=policy_predictions if use_per_label else baseline_predictions

In [ ]:
# L. Per-label evidence and reviewable model choices.
precision,recall,f1,positive_count=precision_recall_fscore_support(y_meta,selected_outer_predictions,zero_division=0)
per_label=pd.DataFrame({"label":LABELS,"positive_count":positive_count,
    "outer_precision":precision,"outer_recall":recall,"outer_f1":f1,
    "selected_candidate":[candidate_names[i] for i in chosen],"final_threshold":final_thresholds,
    "test_predicted_positives":(final_test>=final_thresholds).sum(0)})
per_label.to_csv(RUN_DIR/"per_label_report.csv",index=False)
display(per_label.round(4))
print("All-row fitted macro F1 (optimistic):",macro_score(y_meta,final_valid>=final_thresholds))
pd.DataFrame(final_inner_scores,index=candidate_names,columns=LABELS).to_csv(RUN_DIR/"inner_candidate_scores.csv")
errors=meta_frame[[ID_COL,TEXT_COL]].copy()
errors["wrong_labels"]=(y_meta!=selected_outer_predictions).sum(1)
for j,label in enumerate(LABELS):
    errors[label+"_true"]=y_meta[:,j]
    errors[label+"_pred"]=selected_outer_predictions[:,j]
errors.loc[errors.wrong_labels>0].to_csv(RUN_DIR/"selection_errors.csv",index=False)

In [ ]:
# M. Submission: preserve the association between test text, ID, and prediction.
def build_submission(test_ids,target_ids,columns,probability,thresholds):
    probability=check_probabilities(probability,(len(test_ids),len(LABELS)))
    assert len(set(test_ids))==len(test_ids)
    assert len(target_ids)==len(test_ids) and set(target_ids)==set(test_ids)
    assert len(set(target_ids))==len(target_ids)
    assert len(columns)==len(LABELS)+1 and set(columns)=={ID_COL,*LABELS}
    thresholds=np.asarray(thresholds)
    assert thresholds.shape==(len(LABELS),) and np.isfinite(thresholds).all()
    frame=pd.DataFrame((probability>=thresholds).astype(np.int8),columns=LABELS,
                       index=pd.Index(test_ids,name=ID_COL))
    frame=frame.loc[target_ids].reset_index()[columns]
    assert frame[ID_COL].tolist()==target_ids and frame[LABELS].isin([0,1]).all().all()
    return frame

submission=build_submission(test_df[ID_COL].tolist(),submission_ids,submission_columns,final_test,final_thresholds)
SUBMISSION_PATH=RUN_DIR/"submission_v4.csv"
submission.to_csv(SUBMISSION_PATH,index=False)
roundtrip=pd.read_csv(SUBMISSION_PATH,dtype={ID_COL:str})
assert roundtrip[ID_COL].tolist()==submission_ids
assert np.array_equal(roundtrip[LABELS].to_numpy(),submission[LABELS].to_numpy())
assert roundtrip.shape==(len(test_df),len(LABELS)+1)
print("SUBMIT THIS CSV:",SUBMISSION_PATH)
display(submission.head())

In [ ]:
# N. Save the submission and all labels/probabilities needed for the next review.
report={"version":PIPELINE_VERSION,"mode":MODE,"prior_kaggle":{"v2":.516,"v3":.446},"new_kaggle_score":None,
    "source":source_report,"sample_ids_match":sample_match,"submission_id_policy":SUBMISSION_ID_POLICY,
    "selection":{"outer_baseline_macro_f1":baseline_score,"outer_per_label_macro_f1":policy_score,
        "selected_policy":"per_label" if use_per_label else "baseline",
        "all_row_fitted_macro_f1":macro_score(y_meta,final_valid>=final_thresholds),
        "chosen_candidates":dict(zip(LABELS,[candidate_names[i] for i in chosen])),
        "thresholds":dict(zip(LABELS,final_thresholds.tolist())),
        "minimum_model_gain":MIN_MODEL_SWITCH_GAIN,"minimum_positive_count":MIN_MODEL_SWITCH_POSITIVES,
        "minimum_policy_gain":MIN_POLICY_GAIN,"inner_folds":INNER_FOLDS,"outer_folds":N_FOLDS,
        "bootstrap_samples":BOOTSTRAPS,"outer_choices":outer_choices},
    "limitations":"Development diagnostic: prior V3 model selection and current policy selection reuse labels. No guaranteed improvement over V2."}
(RUN_DIR/"experiment_report.json").write_text(json.dumps(report,indent=2))
np.savez_compressed(RUN_DIR/"all_predictions.npz",base_valid=base_valid,base_test=base_test,
    base_names=np.asarray(base_names),candidate_names=np.asarray(candidate_names),candidate_weights=candidate_weights,
    labels=y_meta,label_names=np.asarray(LABELS),folds=outer_folds,
    selected_valid=final_valid,selected_test=final_test,thresholds=final_thresholds,
    evaluation_ids=meta_frame[ID_COL].to_numpy(dtype=str),test_ids=test_df[ID_COL].to_numpy(dtype=str))
BUNDLE_PATH=RUN_DIR/"v4_results.zip"
with zipfile.ZipFile(BUNDLE_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in RUN_DIR.iterdir():
        if path.is_file() and path.suffix in {".csv",".json",".npz"}:z.write(path,path.name)
print("Ready:",SUBMISSION_PATH,BUNDLE_PATH,sep="\n")
from google.colab import files
files.download(str(SUBMISSION_PATH))
files.download(str(BUNDLE_PATH))

## Interpretation and next step
Submit **submission_v4.csv** once and record the Kaggle score. Send **v4_results.zip** and that score. Do not replace V2 as your best result unless the new submission actually improves on 0.516.

The saved predictions let us inspect threshold failures, compare label-specific models, and plan a targeted new training experiment. A higher local development score does not guarantee a higher Kaggle score. This notebook does not automatically make leaderboard submissions or repeatedly optimize against the public score.

If you switch to `MODE="five_fold"`, enable T4 GPU. This is a separate experiment with five models × six epochs. Reducing training batch size to 4 and increasing accumulation to 4 retains an effective batch size of 16 if GPU memory is insufficient. Completed-fold caches are preserved in Drive; interrupted folds restart.

## What was verified before delivery
Notebook schema and Python syntax; actual V3 ZIP parsing and ID alignment; reconstruction/consistency of saved audit labels; correct positive-label F1; probability thresholds below 0.1; grouped splits; inner/outer fitting boundaries; candidate weights and per-label inference consistency; recovery TF-IDF fitting/caching on synthetic data; submission ID order and results bundle construction. Recovery runs end to end on synthetic CSV fixtures paired with a synthetic results archive. The real supplied V3 audit rows were also used for archive-reader consistency checks.

The full competition CSVs have not been supplied here, so actual 500-row calibration and a new Kaggle score remain unmeasured. Optional GPU training has not been executed here.

## External references used
- [scikit-learn F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html): binary positive-class versus macro averaging; this informed the correction to V3's metric.
- [Precision-recall curve](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html): thresholds come from actual prediction scores.
- [Cross-validation and model selection](https://scikit-learn.org/stable/modules/cross_validation.html): separate model/threshold fitting from evaluation; development reuse limitations remain disclosed.
- [BanglaBERT model card](https://huggingface.co/csebuetnlp/banglabert/blob/main/README.md): model-specific normalization.